In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from google.colab import drive
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

In [21]:
%%capture
device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = "bigscience/bloom-3b"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device)

In [3]:
%%capture
ds = load_dataset("tatsu-lab/alpaca")
ds = ds['train'].remove_columns(['instruction', 'input', 'output'])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [4]:
lora_config = LoraConfig(r=32, target_modules=['query_key_value', 'dense_h_to_4h', 'dense_4h_to_h'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 53,760,000 || all params: 3,056,317,440 || trainable%: 1.7590


In [5]:
def input_format(example):
  enc = tokenizer(
      example['text'],
      max_length=100,
      padding='max_length',
      truncation=True)
  input_ids=enc['input_ids']
  labels=[-100]*len(input_ids)
  template = tokenizer('### Response:')
  ln = len(template['input_ids'])
  for i in range(len(input_ids)-1):
    if input_ids[i:i+ln] == template['input_ids']:
      labels[i+ln:] = input_ids[i+ln:]
      break
  return {'input_ids':input_ids,
          'attention_mask':enc['attention_mask'],
          'labels':labels}

In [6]:
tokenized_ds = ds.map(input_format, remove_columns=['text'])

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [10]:
split_dataset = tokenized_ds.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [17]:
training_args = TrainingArguments(
    output_dir="./best_model",
    learning_rate=2e-5,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,
    warmup_steps=200,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    fp16=True
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

In [ ]:
trainer.train()